<font color="#1aac39" size='6px'><b>Image Smart Redaction (Privacy Protection)</b></font>

<font color="#1aac39" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a privacy-focused pipeline designed to automatically detect and redact sensitive information  with pixel-level precision, ensuring data compliance before analysis.

<font color="#D35400" size='4px'><b>Stage I: Target Identification & Security Configuration</b></font>
Before processing, the system determines *what* constitutes sensitive data in the current context.

**Automatic Security List:** By default, it loads a predefined security protocol (`SMART_REDACTION_OBJECTS`) containing universal privacy targets:
* <font color="#884EA0"><i>Personal Identity:</i></font> "face", "signature", "id card".
* <font color="#884EA0"><i>Financial/Tech:</i></font> "credit card", "license plate", "laptop screen", "mobile phone screen".

**Custom Prompt Mode:** The user can override this by specifying a custom target (e.g., `prompt_text="confidential document"`), focusing the scanner strictly on that object.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Precision Segmentation</b></font>
The system employs the **SAM 3 (Segment Anything Model 3)** to generate high-fidelity masks. Unlike simple bounding boxes (which might redact background context), SAM 3 creates exact contours.

**Text-Prompted Inference:** The identified targets are converted into text prompts.

**Mask Generation:** SAM 3 predicts binary **Spatial Masks** for every instance of the target, capturing fine details (e.g., the irregular shape of a hand holding a card) to ensure the redaction is tight and accurate.

<font color="#D35400" size='4px'><b>Stage III: Filtering & Mask Aggregation</b></font>
To prevent false positives or "over-redaction," the raw predictions undergo refinement.

**Confidence Thresholding:** Masks with low confidence scores (below `SAM3_CONF_IMAGE_THRESHOLD`) are discarded.

**Boolean Aggregation:** If multiple targets are present (e.g., a "face" and a "phone" in the same image), their individual masks are logically combined (OR operation) into a single, unified "Redaction Map."

<font color="#D35400" size='4px'><b>Stage IV: The Redaction Engine (Gaussian Blurring)</b></font>
Once the targets are isolated, the system applies the obscuration layer.

**Kernel Generation:** A heavy Gaussian Blur Kernel is generated (controlled by `blur_strength`, typically 51+).

**Selective Application:** The blur is applied *only* to the pixels flagged in the Redaction Map. The background and non-sensitive objects remain 100% crisp and unaltered, preserving the image's utility for non-sensitive analysis.

<font color="#D35400" size='4px'><b>Stage V: Visualization & Output</b></font>
The final step finalizes the secure image.

**Reconstruction:** The blurred regions are composited back into the original RGB image.

**Compression:** The metadata and segmentation details are compressed and saved via `utils.compress_and_save_output_information` for audit trails or verification.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = False

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "smart_redaction.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*smart_redaction:*</font> Contains the main pipeline for generate privacy masks.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import torch
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import smart_redaction
import visualization

In [ ]:
if COLAB:
    IMAGE_FILENAMES = ["images/g1.png", "images/g2.png"]
    IMAGE_PATH = os.path.join(repo_root, "images", IMAGE_FILENAMES[0])

    if not os.path.exists(IMAGE_PATH):
        alt_path = os.path.join(repo_root, IMAGE_FILENAMES[0])
        if os.path.exists(alt_path):
            VIDEO_PATH = alt_path
        else:
            print(f"      --- Warning: Could not find {IMAGE_FILENAMES[0]} in {os.path.dirname(IMAGE_PATH)}")
else:        
    IMAGE_FILENAMES = ["images/g1.png", "images/g2.png"]

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_IMAGE_FOR_COUNTING = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

In [ ]:
from matplotlib import pyplot as plt
import cv2

# -AI Generated Images-
plt.figure(figsize=(20, 12))
for i, img_file in enumerate(IMAGE_FILENAMES):
    plt.subplot(len(IMAGE_FILENAMES), 1, i+1)
    plt.imshow(plt.imread(img_file))
    plt.tight_layout()
    plt.axis('off')

<font color="#D35400" size='6px'>Execution1: BLIP Automatic Detection</font>

This cell runs the Standard Automatic Mode. By using <font color="#D35400" size='4px'>args.mode='blip'</font>, employ a locally hosted Vision-Language Model to "look" at image and figure out what's inside without needing external APIs.

The function <font color="#D35400"><i>get_classes_blip</i></font> orchestrates a smart scanning process:

<font color="#D35400" size='4px'><i>Smart Cropping:</i></font> To ensure don't miss small details, the image is sliced into multiple strategic crops (zoom-ins) using <font color="#D35400"><i>visualization.get_image_crops</i></font>. This allows the AI to focus on specific regions rather than just the cluttered whole.

<font color="#D35400" size='4px'><i>Local Captioning:</i></font> Runs the <font color="#c5885f"><i>BlipForConditionalGeneration</i></font> model on every single crop. It generates a descriptive sentence for each section,  creating a textual map of the image content.

<font color="#D35400" size='4px'><i>Noun Extraction:</i></font> All these captions are stitched together. then uses a standard NLP library (SpaCy) to parse this text and extract the specific nouns (e.g., "car", "dog"), creating a clean target list for SAM3 to segment.

In [ ]:
redacted_images_rgb = []

for i, img_file in enumerate(IMAGE_FILENAMES):
    output_informations = smart_redaction.smart_redaction(img_file, args, prompt_text="license plate", blur_strength=51)
    redacted_images_rgb.append(output_informations)

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
for output in redacted_images_rgb:
    visualization.show_save_smart_redaction_visualizations(output)

---

<font color="#3c983c" size='6px'>Execution2: LLM-Enhanced Reasoning</font>

This cell activates the LLM Mode. By simply setting <font color="#3c983c" size='4px'>args.mode='llm'</font>, bypass the standard local models and instead use a Multimodal LLM to get a much deeper understanding of the scene.

This function runs on a fast Async Pipeline:

<font color="#CB4335" size='4px'><i>Asyncio Parallelism:</i></font> unlike BLIP (which works sequentially), the function <font color="#CB4335"><i>get_classes_llm</i></font> chops the image into multiple detailed crops using <font color="#CB4335" size='4px'><i>visualization.get_image_crops*</i></font>. It then sends API requests for all those crops at the exact same time, analyzing the whole image in parallel rather than waiting.

<font color="#CB4335" size='4px'><i>Visual Reasoning:</i></font> Instead of a generic "caption", instruct the model to focus strictly on describing physical objects. This allows the system to spot subtle or complex items that standard BLIP models usually miss.

<font color="#CB4335" size='4px'><i>Intelligent Parsing:</i></font> The raw descriptions from the vision model are collected and sent to a second "Parser" LLM. This step, managed by <font color="#CB4335"><i>utils.search_engine_clean_nouns</i></font>, cleans up the text and extracts a neat list of target nouns that act as precise prompts for SAM3.

In [ ]:
redacted_images_rgb = []

for i, img_file in enumerate(IMAGE_FILENAMES):
    output_informations = smart_redaction.smart_redaction(img_file, args, prompt_text="license plate, face, laptop screen, phone screen", blur_strength=51)
    redacted_images_rgb.append(output_informations)

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
for output in redacted_images_rgb:
    visualization.show_save_smart_redaction_visualizations(output)